# Projeto 2: corrigindo o VM2 pela inflação

Função que usa o [`python-bcb`](https://wilsonfreitas.github.io/python-bcb/) para baixar o IPCA
(série 433 do Banco Central) e calcular o fator que leva reais de um ano a reais de outro,
pela média anual do índice.

In [1]:
%pip install -q python-bcb

In [2]:
import numpy as np
import pandas as pd
from bcb import sgs


def fator_ipca(ano_origem, ano_destino):
    """Fator que leva valores em reais de ano_origem para reais de ano_destino."""
    inicio, fim = min(ano_origem, ano_destino), max(ano_origem, ano_destino)
    ipca = sgs.get({'ipca': 433}, start=f'{inicio}-01-01', end=f'{fim}-12-31')
    indice = (1 + ipca['ipca'] / 100).cumprod()
    media_anual = indice.groupby(indice.index.year).mean()
    return media_anual[ano_destino] / media_anual[ano_origem]


fator = fator_ipca(2019, 2025)
fator

np.float64(1.4003352655224188)

## Exemplo: VM2 de 2019 em reais de 2025

In [3]:
# troque pelo DataFrame de vocês
dados = pd.DataFrame({
    'ano': [2019, 2019, 2025, 2025],
    'vm2': [4200.0, 4600.0, 5800.0, 6100.0],
})

dados['vm2_real'] = np.where(dados['ano'] == 2019, dados['vm2'] * fator, dados['vm2'])
dados

,ano,vm2,vm2_real
0,2019,4200.0,5881.408115
1,2019,4600.0,6441.542221
2,2025,5800.0,5800.000000
3,2025,6100.0,6100.000000
